In [ ]:
import pandas as pd
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=AdventureWorks2025;"
    "Trusted_Connection=yes;"
)

query = "SELECT * FROM dbo.vw_fact_sales;"
df = pd.read_sql(query, conn)

print(df.shape)
df.head()

In [ ]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["year"] = df["OrderDate"].dt.year
df["month"] = df["OrderDate"].dt.month
df["quarter"] = df["OrderDate"].dt.quarter
df["day_of_week"] = df["OrderDate"].dt.day_name()

In [ ]:
order_features = (
    df.groupby("SalesOrderID")
      .agg(
          order_revenue=("Revenue", "sum"),
          order_qty=("OrderQty", "sum"),
          unique_products=("ProductID", "nunique"),
          customer_id=("CustomerID", "first"),
          territory=("TerritoryName", "first"),
          order_date=("OrderDate", "first")
      )
      .reset_index()
)

order_features["avg_item_value"] = (
    order_features["order_revenue"] / order_features["order_qty"]
)

In [ ]:
customer_features = (
    df.groupby("CustomerID")
      .agg(
          total_revenue=("Revenue", "sum"),
          total_orders=("SalesOrderID", "nunique"),
          total_qty=("OrderQty", "sum"),
          unique_products=("ProductID", "nunique"),
          first_order_date=("OrderDate", "min"),
          last_order_date=("OrderDate", "max")
      )
      .reset_index()
)

customer_features["aov"] = (
    customer_features["total_revenue"] / customer_features["total_orders"]
)
customer_features["customer_lifespan_days"] = (
    customer_features["last_order_date"] - customer_features["first_order_date"]
).dt.days

In [ ]:
category_features = (
    df.groupby("ProductCategory")
      .agg(
          total_revenue=("Revenue", "sum"),
          orders=("SalesOrderID", "nunique"),
          customers=("CustomerID", "nunique")
      )
      .reset_index()
)

territory_features = (
    df.groupby(["TerritoryGroup", "TerritoryName"])
      .agg(
          total_revenue=("Revenue", "sum"),
          orders=("SalesOrderID", "nunique"),
          customers=("CustomerID", "nunique")
      )
      .reset_index()
)